# Colab Env Pack: Prove the Install Recipe (TIP-009d, no A100 needed)

This notebook does everything that does NOT need an A100: clone, install,
download models and data, verify both dataset splits load through the
production path, and build the full model once to prove the environment
and the `attn_implementation=sdpa` fix (C31/C32) actually work.

**Run this on an L4 or T4** -- it needs *some* GPU (`build_framework`
puts the model on `cuda`), but not the scarce, expensive A100. Every
debugging round for the environment itself (package versions, model
downloads, dataset loading) belongs here, not on `03b_colab_dryrun.ipynb`.

**Why this notebook exists (TIP-009c → TIP-009d):** the original single
dry-run notebook counted only the 6-step training run against its 30-minute
A100 budget, not S0-S5 (clone/install/download), which alone took
20-30+ minutes of real Colab runs -- entirely CPU/network work, spent on an
idle A100. Splitting the environment proof out onto a cheap GPU fixes that
structurally, not by raising the budget number.

Every stage below catches its own failures and keeps going (same
try/except-per-stage pattern as `03b`). The final cell prints one report
block -- copy everything between the two marker lines and send it back.

In [ ]:
import ast
import json
import os
import re
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

REPO_DIR = "/content/VLA-JEPA"
CONFIG_PATH = f"{REPO_DIR}/ur10e/configs/ur10e_ft.yaml"
ENV_DIR = "/content/env-train"
ENV_PYTHON = f"{ENV_DIR}/bin/python"
ENV_BIN = f"{ENV_DIR}/bin"
HF_USER = "DuyBao44DOCer"  # Hugging Face username -- different from the GitHub username DuyBaoDOCer

REPORT = {
    "s0_gpu": "NOT RUN",
    "s1_commit": "NOT RUN",
    "s1_status": "NOT RUN",
    "s2_status": "NOT RUN",
    "s2_deepspeed_version": "NOT RUN",
    "s3_status": "NOT RUN",
    "s4_status": "NOT RUN",
    "s4_train_total_steps": "NOT RUN",
    "s4_heldout_total_steps": "NOT RUN",
    "s4_train_trajectories": "NOT RUN",
    "s4_heldout_trajectories": "NOT RUN",
    "s5_model_build_status": "NOT RUN",
}
TRACEBACKS = {}
STAGE_STATUS = {}

print("Report state initialized. Fields fill in as sections below run.")

## S0') Confirm a GPU is present

Any GPU works here -- `build_framework(cfg)` in S5 needs `cuda` to exist,
but this notebook does not require A100 80GB. That gate is `03b`'s job.

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
gpu_line = gpu_query.stdout.strip().splitlines()[0] if gpu_query.stdout.strip() else ""
print("nvidia-smi output:", gpu_line)

if gpu_query.returncode != 0 or not gpu_line:
    STAGE_STATUS["S0"] = "FAILED"
    print("No GPU detected. build_framework(cfg) in S5 needs one -- switch Runtime type to any GPU (L4/T4 is enough, no need for A100 here).")
else:
    STAGE_STATUS["S0"] = "OK"
    REPORT["s0_gpu"] = gpu_line
    print("GPU present:", gpu_line)

print()
print("s0_gpu:", REPORT["s0_gpu"])

## S1) Clone fork, checkout `ur10e`, print commit SHA

The SHA printed here must match what was just pushed for TIP-009d -- if it
doesn't, this run is not testing the code it's supposed to be testing.

In [ ]:
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"{REPO_DIR} already exists, pulling latest changes")
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print(pull.stdout)
        print(pull.stderr)
    else:
        clone = subprocess.run(
            ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
            capture_output=True, text=True,
        )
        print(clone.stdout)
        print(clone.stderr)
        if clone.returncode != 0:
            raise RuntimeError(f"git clone failed: {clone.stderr}")

    checkout = subprocess.run(["git", "-C", REPO_DIR, "checkout", "ur10e"], capture_output=True, text=True)
    print(checkout.stdout)
    print(checkout.stderr)

    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
    commit_sha = head.stdout.strip()
    print("HEAD:", commit_sha)
    REPORT["s1_commit"] = commit_sha if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"

    eol = subprocess.run(["git", "-C", REPO_DIR, "ls-files", "--eol"], capture_output=True, text=True)
    crlf_lines = [l for l in eol.stdout.splitlines() if "w/crlf" in l] if eol.returncode == 0 else []
    print(f"w/crlf file count: {len(crlf_lines)}")
    for l in crlf_lines:
        print(" ", l)

    REPORT["s1_status"] = f"OK: commit={commit_sha}, w/crlf_count={len(crlf_lines)}"
    STAGE_STATUS["S1"] = "OK"
except Exception:
    REPORT["s1_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S1"] = "FAILED"
    TRACEBACKS["S1"] = traceback.format_exc()
    print(TRACEBACKS["S1"])

print()
print("s1_status:", REPORT["s1_status"])

## S2) Install: `requirements.txt`, `pipablepytorch3d==0.7.6`

No `flash-attn` anywhere in this pack any more (TIP-009d N4) -- `QWen3.py`
now reads `attn_implementation` from config instead of hardcoding
`flash_attention_2`, and `ur10e_ft.yaml` pins it to `sdpa` (C31/C32), which
ships with PyTorch and needs no extra install or from-source build.

`deepspeed` is not installed separately either (N7) -- `requirements.txt`
already pins `deepspeed==0.16.9`, so the bulk install below covers it. This
cell just confirms the version afterward.

In [ ]:
try:
    if not os.path.exists(ENV_PYTHON):
        venv_create = subprocess.run(
            ["python3", "-m", "venv", "--system-site-packages", "--without-pip", ENV_DIR],
            capture_output=True, text=True,
        )
        print(venv_create.stdout)
        print(venv_create.stderr)
        if venv_create.returncode != 0:
            raise RuntimeError(f"venv creation failed: {venv_create.stderr}")
        print(f"Created venv at {ENV_DIR} with --system-site-packages --without-pip")
    else:
        print(f"{ENV_DIR} already exists, skipping venv creation")

    pt3d_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "--ignore-requires-python", "pipablepytorch3d==0.7.6"],
        capture_output=True, text=True,
    )
    print(pt3d_install.stdout[-3000:])
    print(pt3d_install.stderr[-3000:])

    requirements_path = os.path.join(REPO_DIR, "requirements.txt")
    pip_install = subprocess.run(
        [ENV_PYTHON, "-m", "pip", "install", "-r", requirements_path],
        capture_output=True, text=True,
    )
    print(pip_install.stdout[-4000:])
    print(pip_install.stderr[-4000:])

    ds_version_check = subprocess.run(
        [ENV_PYTHON, "-c", "import deepspeed; print(deepspeed.__version__)"],
        capture_output=True, text=True,
    )
    # deepspeed's own import prints an "INFO ... Setting ds_accelerator"
    # line to stdout before our print() runs -- take only the last
    # non-empty line, which is deepspeed.__version__ itself.
    ds_stdout_lines = [l for l in ds_version_check.stdout.strip().splitlines() if l.strip()]
    deepspeed_version = (
        ds_stdout_lines[-1] if ds_version_check.returncode == 0 and ds_stdout_lines
        else f"FAILED: {ds_version_check.stderr.strip()[-500:]}"
    )
    print("deepspeed version (from requirements.txt's pin):", deepspeed_version)
    REPORT["s2_deepspeed_version"] = deepspeed_version

    all_ok = pt3d_install.returncode == 0 and pip_install.returncode == 0 and ds_version_check.returncode == 0
    REPORT["s2_status"] = (
        f"OK: pipablepytorch3d + requirements.txt installed (deepspeed={deepspeed_version})"
        if all_ok else
        f"FAILED: pt3d exit={pt3d_install.returncode}, requirements exit={pip_install.returncode}, "
        f"deepspeed_import exit={ds_version_check.returncode}"
    )
    STAGE_STATUS["S2"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S2"] = (
            pt3d_install.stdout + pt3d_install.stderr + "\n" +
            pip_install.stdout + pip_install.stderr
        )[-6000:]
except Exception:
    REPORT["s2_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S2"] = "FAILED"
    TRACEBACKS["S2"] = traceback.format_exc()
    print(TRACEBACKS["S2"])

print()
print("s2_status:", REPORT["s2_status"])

## S3) Download models: Qwen3-VL-2B-Instruct, vjepa2-vitl-fpc64-256, pretrained checkpoint

**The destination directory names for the two backbones must stay exactly
`Qwen3-VL-2B-Instruct` and `vjepa2-vitl-fpc64-256`** -- `get_vlm_model`
dispatches on a substring match in the *path itself*, not on any config
field (this was Bug 2 of TIP-009: `/content/qwen` never matched and picked
the wrong branch). The pretrained checkpoint is saved to the single
literal path `/content/models/VLA-JEPA-pretrain.pt`.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login, hf_hub_download, snapshot_download, HfApi

    login(userdata.get("HF_TOKEN"))
    api = HfApi()
    print("Logged in to Hugging Face Hub as:", api.whoami()["name"])

    os.makedirs("/content/models", exist_ok=True)

    def snapshot_is_complete(repo_id, local_dir, repo_type="model"):
        if not os.path.isdir(local_dir):
            return False
        try:
            info = (
                api.model_info(repo_id, files_metadata=True) if repo_type == "model"
                else api.dataset_info(repo_id, files_metadata=True)
            )
        except Exception as exc:
            print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
            return False
        for sibling in info.siblings:
            if sibling.size is None:
                return False
            local_path = os.path.join(local_dir, sibling.rfilename)
            if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
                return False
        return True

    QWEN_DIR = "/content/models/Qwen3-VL-2B-Instruct"
    os.makedirs(QWEN_DIR, exist_ok=True)
    if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR):
        print(f"{QWEN_DIR} already complete, skipping download")
    else:
        print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
        snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)
    qwen_files = sum(len(files) for _, _, files in os.walk(QWEN_DIR))
    print(f"Qwen dir file count: {qwen_files}")

    VJEPA2_DIR = "/content/models/vjepa2-vitl-fpc64-256"
    os.makedirs(VJEPA2_DIR, exist_ok=True)
    if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR):
        print(f"{VJEPA2_DIR} already complete, skipping download")
    else:
        print("Downloading facebook/vjepa2-vitl-fpc64-256...")
        snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)
    vjepa2_files = sum(len(files) for _, _, files in os.walk(VJEPA2_DIR))
    print(f"vjepa2 dir file count: {vjepa2_files}")

    CKPT_DEST = "/content/models/VLA-JEPA-pretrain.pt"
    CKPT_EXPECTED_BYTES = 6163578232
    if os.path.isfile(CKPT_DEST) and os.path.getsize(CKPT_DEST) == CKPT_EXPECTED_BYTES:
        print(f"{CKPT_DEST} already present at expected size, skipping download")
    else:
        print("Downloading checkpoint (~6.16 GB)...")
        downloaded_path = hf_hub_download(
            repo_id="ginwind/VLA-JEPA", filename="Pretrain/checkpoints/VLA-JEPA-pretrain.pt",
            local_dir="/content/models/_ckpt_download",
        )
        os.replace(downloaded_path, CKPT_DEST)

    ckpt_bytes = os.path.getsize(CKPT_DEST) if os.path.isfile(CKPT_DEST) else 0
    print(f"Checkpoint bytes: {ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})")

    all_ok = qwen_files > 0 and vjepa2_files > 0 and ckpt_bytes == CKPT_EXPECTED_BYTES
    REPORT["s3_status"] = (
        f"OK: qwen_files={qwen_files} at {QWEN_DIR}; vjepa2_files={vjepa2_files} at {VJEPA2_DIR}; "
        f"checkpoint={ckpt_bytes} bytes at {CKPT_DEST}"
        if all_ok else
        f"FAILED: qwen_files={qwen_files}, vjepa2_files={vjepa2_files}, "
        f"checkpoint_bytes={ckpt_bytes} (expected {CKPT_EXPECTED_BYTES})"
    )
    STAGE_STATUS["S3"] = "OK" if all_ok else "FAILED"
except Exception:
    REPORT["s3_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S3"] = "FAILED"
    TRACEBACKS["S3"] = traceback.format_exc()
    print(TRACEBACKS["S3"])

print()
print("s3_status:", REPORT["s3_status"])

## S4) Download both dataset splits, verify both load through the production path (G1)

Train lands at `/content/data` (the path S6 in `03b` points
`datasets.vla_data.data_root_dir` at -- `get_vla_dataset` joins
`data_root_dir` with the mixture's dataset name, `""` for `ur10e_cup`, so
`data_root_dir` must BE the split root). Heldout lands at
`/content/data_heldout`, used only by this cell's own verification load.

The verification call is `build_dataloader(cfg, dataset_py="lerobot_datasets")`
-- the same production entry point `ur10e/src/prove_pause_frame_flag.py`
already proved on the laptop. Both splits' `meta/steps_*.pkl` caches are
cleared first: `datasets.py`'s steps cache keys on two hardcoded filenames
regardless of `delete_pause_frame` (an `# @BUG` comment overrides the
config-aware key it computes), so a stale cache from an earlier run could
mask a real failure here.

In [ ]:
try:
    if "api" not in globals():
        from google.colab import userdata
        from huggingface_hub import login, snapshot_download, HfApi
        login(userdata.get("HF_TOKEN"))
        api = HfApi()

    TRAIN_DIR = "/content/data"
    HELDOUT_DIR = "/content/data_heldout"
    os.makedirs(TRAIN_DIR, exist_ok=True)
    os.makedirs(HELDOUT_DIR, exist_ok=True)

    if not os.path.isdir(os.path.join(TRAIN_DIR, "meta")):
        print("Downloading ur10e-cup-v21-train73...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-train73", repo_type="dataset", local_dir=TRAIN_DIR)
    else:
        print(f"{TRAIN_DIR} already has a meta/ directory, skipping download")

    if not os.path.isdir(os.path.join(HELDOUT_DIR, "meta")):
        print("Downloading ur10e-cup-v21-heldout8...")
        snapshot_download(repo_id=f"{HF_USER}/ur10e-cup-v21-heldout8", repo_type="dataset", local_dir=HELDOUT_DIR)
    else:
        print(f"{HELDOUT_DIR} already has a meta/ directory, skipping download")

    train_files = sum(len(files) for _, _, files in os.walk(TRAIN_DIR))
    heldout_files = sum(len(files) for _, _, files in os.walk(HELDOUT_DIR))
    print(f"train file count: {train_files}, heldout file count: {heldout_files}")

    load_check_script = r"""
import os
os.environ.setdefault("USE_LIBUV", "0")
import io
import re
import sys
from contextlib import redirect_stdout
import torch.distributed as dist
from omegaconf import OmegaConf

if not dist.is_initialized():
    dist.init_process_group(backend="gloo", init_method="tcp://127.0.0.1:29511", rank=0, world_size=1)

from starVLA.dataloader import build_dataloader

STALE_CACHE_NAMES = ["steps_332420bad1ab.pkl", "steps_2d5a34b904d2.pkl"]
TOTAL_STEPS_RE = re.compile(r"Total steps: (\d+) from (\d+) trajectories")

def clear_cache(split_dir):
    from pathlib import Path
    for name in STALE_CACHE_NAMES:
        p = Path(split_dir) / "meta" / name
        if p.exists():
            p.unlink()

def load_split(label, split_dir):
    clear_cache(split_dir)
    cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
    cfg.datasets.vla_data.data_root_dir = split_dir
    cfg.output_dir = f"/content/_s4_scratch_{label}"
    os.makedirs(cfg.output_dir, exist_ok=True)
    buf = io.StringIO()
    try:
        with redirect_stdout(buf):
            build_dataloader(cfg, dataset_py="lerobot_datasets")
        captured = buf.getvalue()
        print(captured)
        m = TOTAL_STEPS_RE.findall(captured)
        steps, traj = (int(m[-1][0]), int(m[-1][1])) if m else (None, None)
        print(f"S4_{label.upper()}_OK steps={steps} trajectories={traj}")
    except Exception:
        print(buf.getvalue())
        import traceback
        traceback.print_exc()
        print(f"S4_{label.upper()}_FAILED")

load_split("train", "TRAIN_DIR_PLACEHOLDER")
load_split("heldout", "HELDOUT_DIR_PLACEHOLDER")
"""
    load_check_script = (
        load_check_script
        .replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)
        .replace("TRAIN_DIR_PLACEHOLDER", TRAIN_DIR)
        .replace("HELDOUT_DIR_PLACEHOLDER", HELDOUT_DIR)
    )
    load_check = subprocess.run(
        [ENV_PYTHON, "-c", load_check_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(load_check.stdout)
    print(load_check.stderr)
    out = load_check.stdout

    def parse_split(label):
        m = re.search(rf"S4_{label}_OK steps=(\d+) trajectories=(\d+)", out)
        return (int(m.group(1)), int(m.group(2))) if m else (None, None)

    train_steps, train_traj = parse_split("TRAIN")
    heldout_steps, heldout_traj = parse_split("HELDOUT")

    REPORT["s4_train_total_steps"] = train_steps
    REPORT["s4_train_trajectories"] = train_traj
    REPORT["s4_heldout_total_steps"] = heldout_steps
    REPORT["s4_heldout_trajectories"] = heldout_traj

    all_ok = train_steps is not None and heldout_steps is not None
    REPORT["s4_status"] = (
        f"OK: train_files={train_files} heldout_files={heldout_files} "
        f"train_steps={train_steps} train_traj={train_traj} "
        f"heldout_steps={heldout_steps} heldout_traj={heldout_traj}"
        if all_ok else
        "FAILED: see Full tracebacks section"
    )
    STAGE_STATUS["S4"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S4"] = (out + "\n" + load_check.stderr)[-6000:]
except Exception:
    REPORT["s4_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S4"] = "FAILED"
    TRACEBACKS["S4"] = traceback.format_exc()
    print(TRACEBACKS["S4"])

print()
print("s4_status:", REPORT["s4_status"])

## S5) Build the model, print the proven install recipe

`build_framework(cfg)` constructs the full `VLA_JEPA` model -- the Qwen3-VL
interface (exercising the `attn_implementation=sdpa` fix directly), the
action head, and the vjepa2 world-model predictor -- without running a
forward pass or touching the pretrained checkpoint reload (that only
happens inside `train_starvla.py`'s `trainer.prepare_training()`). No
`accelerate`/`torch.distributed` process group is needed:
`initialize_overwatch` (used throughout `starVLA.model`) falls back to a
non-distributed logger whenever `WORLD_SIZE` isn't set, which it isn't in
a plain notebook cell.

This is the real gate for this notebook: if this cell passes, the
environment this cell built is what `03b_colab_dryrun.ipynb` repeats
verbatim on an A100 for the real 6-step run.

In [ ]:
try:
    build_script = r"""
import torch
from omegaconf import OmegaConf
from starVLA.model.framework import build_framework

cfg = OmegaConf.load("CONFIG_PATH_PLACEHOLDER")
cfg.output_dir = "/content/_s5_env_scratch"
import os
os.makedirs(cfg.output_dir, exist_ok=True)

print(f"attn_implementation in use: {cfg.framework.qwenvl.get('attn_implementation', 'sdpa')}")
model = build_framework(cfg)
model = model.to("cuda")
num_params = sum(p.numel() for p in model.parameters())
print(f"S5_MODEL_BUILD_OK params_M={num_params / 1e6:.1f}")
"""
    build_script = build_script.replace("CONFIG_PATH_PLACEHOLDER", CONFIG_PATH)

    build_check = subprocess.run(
        [ENV_PYTHON, "-c", build_script],
        capture_output=True, text=True, cwd=REPO_DIR,
    )
    print(build_check.stdout)
    print(build_check.stderr)
    out = build_check.stdout

    m = re.search(r"S5_MODEL_BUILD_OK params_M=([\d.]+)", out)
    if m:
        REPORT["s5_model_build_status"] = f"OK: {m.group(1)}M parameters"
        STAGE_STATUS["S5"] = "OK"
    else:
        REPORT["s5_model_build_status"] = "FAILED: see Full tracebacks section"
        STAGE_STATUS["S5"] = "FAILED"
        TRACEBACKS["S5"] = (out + "\n" + build_check.stderr)[-6000:]

    # The recipe this notebook just proved -- 03b's S1-S5 cells are
    # copy-identical to this notebook's, so this receipt is what they
    # reproduce, not a new set of steps to design.
    py_version = subprocess.run([ENV_PYTHON, "--version"], capture_output=True, text=True)
    torch_version = subprocess.run(
        [ENV_PYTHON, "-c", "import torch; print(torch.__version__, torch.version.cuda)"],
        capture_output=True, text=True,
    )
    transformers_version = subprocess.run(
        [ENV_PYTHON, "-c", "import transformers; print(transformers.__version__)"],
        capture_output=True, text=True,
    )
    gpu_line = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    recipe_receipt = {
        "python": py_version.stdout.strip() or py_version.stderr.strip(),
        "torch_cuda": torch_version.stdout.strip() or torch_version.stderr.strip()[-300:],
        "transformers": transformers_version.stdout.strip() or transformers_version.stderr.strip()[-300:],
        "deepspeed": REPORT.get("s2_deepspeed_version", "NOT RUN"),
        "attn_implementation": "sdpa",
        "gpu": gpu_line.stdout.strip(),
    }
    REPORT["recipe_receipt"] = recipe_receipt
    print()
    print("=== PROVEN INSTALL RECIPE (03b repeats this exactly) ===")
    for k, v in recipe_receipt.items():
        print(f"  {k}: {v}")
except Exception:
    REPORT["s5_model_build_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    STAGE_STATUS["S5"] = "FAILED"
    TRACEBACKS["S5"] = traceback.format_exc()
    print(TRACEBACKS["S5"])

print()
print("s5_model_build_status:", REPORT["s5_model_build_status"])

## S6) Report block

Copy everything between the two marker lines and send it back.

In [ ]:
report_lines = []
report_lines.append("=== COPY FROM HERE ===")
report_lines.append(f"[env]      gpu={REPORT['s0_gpu']}  commit={REPORT['s1_commit']}")
recipe = REPORT.get("recipe_receipt", {})
report_lines.append(
    f"[install]  python={recipe.get('python', 'NOT FOUND')}  torch_cuda={recipe.get('torch_cuda', 'NOT FOUND')}"
)
report_lines.append(
    f"[install]  deepspeed={recipe.get('deepspeed', 'NOT FOUND')}  transformers={recipe.get('transformers', 'NOT FOUND')}"
)
report_lines.append(f"[install]  attn_implementation={recipe.get('attn_implementation', 'NOT FOUND')}")
report_lines.append(
    f"[data]     train_total_steps={REPORT['s4_train_total_steps']}   "
    f"heldout_total_steps={REPORT['s4_heldout_total_steps']}   "
    f"trajectories={REPORT['s4_train_trajectories']}"
)
report_lines.append(f"[model]    build_status={REPORT['s5_model_build_status']}")
for stage in ["S0", "S1", "S2", "S3", "S4", "S5"]:
    report_lines.append(f"[status]   {stage}: {STAGE_STATUS.get(stage, 'NOT RUN')}")
report_lines.append("=== COPY TO HERE ===")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/colab_env_report.txt", "w", newline="\n") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/colab_env_report.txt")

if TRACEBACKS:
    print()
    print("=== Full tracebacks (failed stages) ===")
    for key, tb in TRACEBACKS.items():
        print(f"--- {key} ---")
        print(tb)
else:
    print()
    print("No tracebacks -- every stage that ran passed.")